### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

Probamos con una palbra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## 1. Vectorización y similaridad de documentos

In [ ]:
np.random.seed(42)
random_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)

for doc_idx in random_indices:
    sim = cosine_similarity(X_train[doc_idx], X_train)[0]
    top5_idx = np.argsort(sim)[::-1][1:6]
    clase_orig = newsgroups_train.target_names[y_train[doc_idx]]
    clases_sim = [newsgroups_train.target_names[y_train[i]] for i in top5_idx]
    sims_vals = [round(sim[i], 3) for i in top5_idx]

    print(f"Documento {doc_idx} - Clase: {clase_orig}")
    print(f"Texto: {newsgroups_train.data[doc_idx][:200].strip()!r}")
    print("5 más similares:")
    for c, s in zip(clases_sim, sims_vals):
        print(f"  {c} (sim={s})")
    print()

En la mayoría de los casos, los 5 documentos más similares pertenecen a la misma clase que el documento original. Esto tiene sentido porque textos del mismo tema comparten vocabulario específico, lo que resulta en vectores TF-IDF con alta similaridad coseno. En algunos casos aparecen documentos de clases distintas pero temáticamente relacionadas (por ejemplo, `talk.religion.misc` y `soc.religion.christian`), lo que también es esperable dado que comparten términos religiosos. Cuando el documento original es muy corto o genérico, los similares tienden a ser de clases más variadas.

## 2. Clasificación por prototipos

In [ ]:
sim_matrix = cosine_similarity(X_test, X_train)
best_train_idx = np.argmax(sim_matrix, axis=1)
y_pred_proto = y_train[best_train_idx]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-score macro (clasificación por prototipos): {f1_proto:.4f}")

El modelo de prototipos logra clasificar asignando a cada documento de test la clase del documento de entrenamiento más similar. El resultado es inferior al de Naive Bayes, lo cual es esperable ya que este método no aprende ninguna distribución estadística sino que simplemente busca el vecino más cercano. La ventaja es que no requiere entrenamiento, pero su rendimiento depende de qué tan representativos sean los documentos individuales del conjunto de entrenamiento. Cuando el documento más similar es de la clase incorrecta, el error se propaga directamente.

## 3. Modelos Naive Bayes

In [ ]:
experimentos = [
    (TfidfVectorizer(),                                    MultinomialNB(),        "TfidfVectorizer() + MultinomialNB()"),
    (TfidfVectorizer(min_df=2),                            MultinomialNB(),        "TfidfVectorizer(min_df=2) + MultinomialNB()"),
    (TfidfVectorizer(sublinear_tf=True),                   MultinomialNB(),        "TfidfVectorizer(sublinear_tf=True) + MultinomialNB()"),
    (TfidfVectorizer(),                                    ComplementNB(),         "TfidfVectorizer() + ComplementNB()"),
    (TfidfVectorizer(min_df=2),                            ComplementNB(),         "TfidfVectorizer(min_df=2) + ComplementNB()"),
    (TfidfVectorizer(sublinear_tf=True),                   ComplementNB(),         "TfidfVectorizer(sublinear_tf=True) + ComplementNB()"),
    (TfidfVectorizer(min_df=2, sublinear_tf=True),         ComplementNB(),         "TfidfVectorizer(min_df=2, sublinear_tf=True) + ComplementNB()"),
    (TfidfVectorizer(min_df=2, sublinear_tf=True),         ComplementNB(alpha=0.1),"TfidfVectorizer(min_df=2, sublinear_tf=True) + ComplementNB(alpha=0.1)"),
    (TfidfVectorizer(min_df=3, max_df=0.7),                ComplementNB(alpha=0.1),"TfidfVectorizer(min_df=3, max_df=0.7) + ComplementNB(alpha=0.1)"),
]

best_f1 = 0
best_nombre = ""

for vect, clf_nb, nombre in experimentos:
    X_tr = vect.fit_transform(newsgroups_train.data)
    X_te = vect.transform(newsgroups_test.data)
    clf_nb.fit(X_tr, y_train)
    y_p = clf_nb.predict(X_te)
    f1 = f1_score(y_test, y_p, average='macro')
    print(f"{nombre}: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_nombre = nombre

print(f"\nMejor configuración: {best_nombre}")
print(f"F1-score macro: {best_f1:.4f}")

ComplementNB supera a MultinomialNB en todas las configuraciones. Esto se debe a que ComplementNB modela el complemento de cada clase en lugar de la clase directamente, lo que lo hace más robusto cuando el dataset tiene clases desbalanceadas o cuando los documentos son cortos. El parámetro `sublinear_tf=True` también mejora los resultados porque aplica una transformación logarítmica a las frecuencias de términos, reduciendo el peso excesivo de palabras muy frecuentes. Usar `min_df=2` ayuda a eliminar términos que aparecen en un solo documento (probablemente errores tipográficos o nombres propios), reduciendo el ruido. La combinación de estos tres parámetros con un `alpha` bajo en ComplementNB obtiene el mejor F1.

## 4. Similaridad entre palabras

In [ ]:
X_train_T = X_train.T

palabras = ['car', 'god', 'space', 'gun', 'computer']

for palabra in palabras:
    word_idx = tfidfvect.vocabulary_[palabra]
    word_vec = X_train_T[word_idx]
    sim = cosine_similarity(word_vec, X_train_T)[0]
    top5_idx = np.argsort(sim)[::-1][1:6]
    similares = [idx2word[i] for i in top5_idx]
    print(f"'{palabra}' -> {similares}")

Al transponer la matriz documento-término obtenemos vectores para cada palabra, donde cada dimensión representa un documento. Dos palabras son similares si tienden a aparecer en los mismos documentos. Los resultados tienen mucho sentido semántico: las palabras más similares a 'car' son términos del ámbito automotor, las de 'god' son términos religiosos, las de 'space' aparecen en discusiones de astronomía y la NASA, las de 'gun' en temas de legislación sobre armas, y las de 'computer' en discusiones técnicas. Esto demuestra que incluso una representación tan simple como TF-IDF captura relaciones semánticas entre palabras a través del contexto de los documentos donde aparecen.